## SAR Imagery Import

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages])

pip_install(
    "torch",
    "torchvision",
    "pandas",
    "numpy",
    "scikit-learn",
    "Pillow",
    "gdown",
    "matplotlib",
    # "wandb",  # uncomment to enable experiment tracking
)

In [ ]:
# import wandb
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import gdown

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/19jLMSzHChVLk-vVAg2muNN2OALzksWob"

# Local directory to download into — sits next to this notebook
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "classification"

# Download from Google Drive if CSVs are not already present
if not (DATA_DIR / "opensar1_labels.csv").exists():
    print("Downloading classification folder from Google Drive...")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    gdown.download_folder(DRIVE_FOLDER_URL, output=str(DATA_DIR), quiet=False, use_cookies=False)
    print("Download complete.")
else:
    print(f"Data already present at {DATA_DIR}, skipping download.")

# Load all three label CSVs
opensar1 = pd.read_csv(DATA_DIR / "opensar1_labels.csv")
opensar2 = pd.read_csv(DATA_DIR / "opensar2_labels.csv")
fusar    = pd.read_csv(DATA_DIR / "fusar_labels.csv")

# Resolve image paths to absolute local paths
opensar1["path"] = opensar1["path"].apply(lambda p: str(PROJECT_ROOT / p))
opensar2["path"] = opensar2["path"].apply(lambda p: str(PROJECT_ROOT / p))
fusar["path"]    = fusar["path"].apply(lambda p: str(PROJECT_ROOT / p))

# print stats
print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

fusar.head()

## Label Mapping and Filtering

In [ ]:
# Build mapping from all unique labels across all three datasets
unique_labels = sorted(set(opensar1['label']).union(
                       set(opensar2['label'])).union(
                       set(fusar['label'])))

label_to_int = {label: i for i, label in enumerate(unique_labels)}

# Apply mapping independently
opensar1.loc[:, 'label_id'] = opensar1['label'].map(label_to_int)
opensar2.loc[:, 'label_id'] = opensar2['label'].map(label_to_int)
fusar.loc[:, 'label_id']    = fusar['label'].map(label_to_int)

print("Label mapping:", label_to_int)

# drop unknown values of the datasets
opensar1 = opensar1[opensar1['label'] != 'unknown']
opensar2 = opensar2[opensar2['label'] != 'unknown']
fusar    = fusar[fusar['label'] != 'unknown']

# length after creation
print(f"OpenSARShip 1: {len(opensar1)} rows")
print(f"OpenSARShip 2: {len(opensar2)} rows")
print(f"FuSARShip:     {len(fusar)} rows")

Label mapping: {'cargo': 0, 'engineering': 1, 'other': 2, 'passenger': 3, 'tanker': 4, 'unknown': 5}
OpenSARShip 1: 10602 rows
OpenSARShip 2: 16280 rows
FuSARShip:     5101 rows


## Dataset Creation

In [ ]:
# create the dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

class create_dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        try:
            image = Image.open(row['path']).convert('L')  # load as grayscale
        except (FileNotFoundError, OSError) as e:
            raise RuntimeError(f"Error loading image at index {idx}: {row['path']}") from e

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(row['label_id'], dtype=torch.long)


# ImageNet normalization values expected by pretrained ResNet
RESNET_MEAN = [0.485, 0.456, 0.406]
RESNET_STD  = [0.229, 0.224, 0.225]

# Training transform — includes augmentation
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15, fill=0),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD)
])

# Validation/test transform — no augmentation
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=RESNET_MEAN, std=RESNET_STD)
])

# Create two versions of each dataset: one for train, one for val/test
datasets_train = {
    'os1'  : create_dataset(opensar1, transform=transform_train),
    'os2'  : create_dataset(opensar2, transform=transform_train),
    'fusar': create_dataset(fusar,    transform=transform_train),
}
datasets_val = {
    'os1'  : create_dataset(opensar1, transform=transform_val),
    'os2'  : create_dataset(opensar2, transform=transform_val),
    'fusar': create_dataset(fusar,    transform=transform_val),
}

print(f"OpenSARShip 1: {len(datasets_train['os1'])} samples")
print(f"OpenSARShip 2: {len(datasets_train['os2'])} samples")
print(f"FuSARShip:     {len(datasets_train['fusar'])} samples")

OpenSARShip 1: 10602 samples
OpenSARShip 2: 16280 samples
FuSARShip:     5101 samples


## Train Test Val Split

In [ ]:
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split

domains = {
    'os1'  : {'df': opensar1},
    'os2'  : {'df': opensar2},
    'fusar': {'df': fusar},
}

for key in domains:
    labels = domains[key]['df']['label_id'].values

    # First split off 20% as final test set
    trainval_idx, test_idx = train_test_split(
        range(len(labels)),
        test_size=0.2,
        stratify=labels,
        random_state=42
    )

    # Then split remaining 80% into 90% train / 10% val
    train_idx, val_idx = train_test_split(
        trainval_idx,
        test_size=0.1,
        stratify=labels[trainval_idx],
        random_state=42
    )

    # Train uses augmented transform; val/test use clean transform
    domains[key]['train_dataset'] = Subset(datasets_train[key], train_idx)
    domains[key]['val_dataset']   = Subset(datasets_val[key],   val_idx)
    domains[key]['test_dataset']  = Subset(datasets_val[key],   test_idx)

    print(f"Domain: {key}")
    print(f"  Train:      {len(train_idx)}")
    print(f"  Validation: {len(val_idx)}")
    print(f"  Test:       {len(test_idx)}")


Domain: os1
  Train:      7632
  Validation: 849
  Test:       2121
Domain: os2
  Train:      11721
  Validation: 1303
  Test:       3256
Domain: fusar
  Train:      3672
  Validation: 408
  Test:       1021


In [ ]:
batch_size = 32

for key in domains:
    domains[key]['train_loader'] = DataLoader(domains[key]['train_dataset'], batch_size=batch_size, shuffle=True)
    domains[key]['val_loader']   = DataLoader(domains[key]['val_dataset'],   batch_size=batch_size, shuffle=False)
    domains[key]['test_loader']  = DataLoader(domains[key]['test_dataset'],  batch_size=batch_size, shuffle=False)

    print(f"{key} — train: {len(domains[key]['train_loader'])} batches | "
          f"val: {len(domains[key]['val_loader'])} batches | "
          f"test: {len(domains[key]['test_loader'])} batches")

os1 — train: 239 batches | val: 27 batches | test: 67 batches
os2 — train: 367 batches | val: 41 batches | test: 102 batches
fusar — train: 115 batches | val: 13 batches | test: 32 batches


## Model Training

In [ ]:
# import wandb
import torch
import torch.nn as nn
from torch.optim import Adam


def train_model(model, model_name, class_names, train_loader, val_loader, epochs=10, lr=1e-3):
    # wandb.init(
    #     project="SAR Project",
    #     name=model_name,
    #     config={
    #         "model": model_name,
    #         "epochs": epochs,
    #         "learning_rate": lr,
    #         "optimizer": "Adam"
    #     }
    # )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_val_acc = 0

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total

        model.eval()
        val_loss, correct, total = 0, 0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss = val_loss / len(val_loader)
        val_acc = 100. * correct / total

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{model_name}_best.pth")

        # wandb.log({
        #     "epoch": epoch + 1,
        #     "train_loss": train_loss,
        #     "val_loss": val_loss,
        #     "train_acc": train_acc,
        #     "val_acc": val_acc,
        # })

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.2f}%")

    # wandb.log({
    #     "confusion_matrix": wandb.plot.confusion_matrix(
    #         probs=None,
    #         y_true=all_labels,
    #         preds=all_preds,
    #         class_names=class_names
    #     )
    # })

    history = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "train_acc": train_accs,
        "val_acc": val_accs
    }

    # wandb.finish()

    return model, history

In [ ]:
from torch.nn import Module, CrossEntropyLoss, Linear
from torchvision import models

class_names = [k for k, v in sorted(label_to_int.items(), key=lambda x: x[1]) if k != 'unknown']
print("class_names:", class_names)

class CNN_resnet_transfer(Module):
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        num_features = self.resnet.fc.in_features
        self.resnet.fc = Linear(num_features, len(class_names))

        for param in self.resnet.parameters():
            param.requires_grad = False

        for name, param in self.resnet.named_parameters():
            if "layer3" in name or "layer4" in name or "fc" in name:
                param.requires_grad = True

    def forward(self, X):
        return self.resnet(X)

In [ ]:
def sanity_check(model, model_name, train_loader, val_loader, num_batches=20):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = CrossEntropyLoss()
    optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

    model.train()
    for i, (inputs, labels) in enumerate(train_loader):
        if i >= num_batches:
            break
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        print(f"[{model_name}] Batch {i+1}/{num_batches} | Loss: {loss.item():.4f}")

In [ ]:
for key in domains:

    domains[key]['model'] = CNN_resnet_transfer()

    sanity_check(domains[key]['model'], key, domains[key]['train_loader'], domains[key]['val_loader'], num_batches = 5)

[os1] Batch 1/5 | Loss: 2.7292
[os1] Batch 2/5 | Loss: 2.4947
[os1] Batch 3/5 | Loss: 2.4151
[os1] Batch 4/5 | Loss: 2.3402
[os1] Batch 5/5 | Loss: 2.4005
[os2] Batch 1/5 | Loss: 2.7131
[os2] Batch 2/5 | Loss: 2.4984
[os2] Batch 3/5 | Loss: 2.5060
[os2] Batch 4/5 | Loss: 2.5057
[os2] Batch 5/5 | Loss: 2.3445
[fusar] Batch 1/5 | Loss: 3.2463
[fusar] Batch 2/5 | Loss: 3.1244
[fusar] Batch 3/5 | Loss: 2.8254
[fusar] Batch 4/5 | Loss: 2.6863
[fusar] Batch 5/5 | Loss: 2.7604


In [ ]:
for key in domains:
    domains[key]['model'] = CNN_resnet_transfer()
    domains[key]['trained_model'], domains[key]['history'] = train_model(
        domains[key]['model'], key, class_names,
        domains[key]['train_loader'], domains[key]['val_loader'],
        epochs=10, lr=1e-4
    )

In [ ]:
import matplotlib.pyplot as plt

def plot_history(histories):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for name, h in histories.items():
        ax1.plot(h["train_loss"], linestyle="dashed", label=f"{name} train")
        ax1.plot(h["val_loss"], label=f"{name} val")
        ax2.plot(h["train_acc"], linestyle="dashed", label=f"{name} train")
        ax2.plot(h["val_acc"], label=f"{name} val")

    ax1.set_title("Loss")
    ax1.set_xlabel("Epoch")
    ax1.legend()

    ax2.set_title("Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.legend()

    plt.tight_layout()
    plt.show()

histories = {}

for key in domains:
    histories[key] = domains[key]['history']
plot_history(histories)